# 03 — Pre-trained Essentia MTG-Jamendo Instrument Embedding & Tagging

This notebook extracts **high-level instrument representations** for music tracks using the official **Essentia MTG-Jamendo Pre-trained MusiCNN Model**.

### Key Capabilities:
1. **40-d Instrument Probabilities:** Zero-shot / pre-trained multi-label predictions across all 40 MTG-Jamendo instrument tags.
2. **200-d Dense Instrument Embeddings:** Deep timbral and acoustic features extracted from the penultimate layer of MusiCNN.
3. **Zero GPU Training Required:** Ready-to-use frozen weights pre-trained on 55k+ tracks.
4. **Direct Integration with Stage 2:** Outputs saved in `features/instrument/` for seamless loading into the Cross-Concept Attention Fusion model (`07_fusion_genre_classifier.ipynb`).


## 1. Install & Import Dependencies

In [ ]:
# Install essentia with tensorflow support, along with scientific stack
!pip install -q essentia-tensorflow scikit-learn matplotlib seaborn tqdm pandas


In [ ]:
import os
import sys
import json
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

try:
    import essentia
    import essentia.standard as es
    print(f"Essentia successfully loaded (Version: {essentia.__version__})")
except ImportError:
    print("WARNING: essentia-tensorflow not found. Run '!pip install essentia-tensorflow' above.")


## 2. Directory & Path Configuration

In [ ]:
# Set root directory (works locally or in Google Colab/Kaggle)
ROOT = Path(os.environ.get("MTG_ROOT", Path.cwd() / "data" / "MTG_Instrument")).resolve()

AUDIO_DIR = ROOT / "dataset" / "audio"
MODELS_DIR = ROOT / "models" / "essentia"
FEAT_DIR = ROOT / "features" / "instrument"
ANN_DIR = ROOT / "annotations"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"

for d in [MODELS_DIR, FEAT_DIR, ANN_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT Directory:   ", ROOT)
print("Features Target:  ", FEAT_DIR)
print("Models Directory: ", MODELS_DIR)


## 3. Download Essentia MTG-Jamendo Pre-trained Model Weights

We download the official TensorFlow frozen graph (`.pb`) and class metadata (`.json`) for the MTG-Jamendo MusiCNN Instrument model from the official Essentia Model Zoo.

In [ ]:
MODEL_BASE_URL = "https://essentia.upf.edu/models/autotagging/mtg_jamendo_instrument/"
MODEL_PB_NAME = "mtg_jamendo_instrument-musiconn-2.pb"
MODEL_JSON_NAME = "mtg_jamendo_instrument-musiconn-2.json"

pb_path = MODELS_DIR / MODEL_PB_NAME
json_path = MODELS_DIR / MODEL_JSON_NAME

def download_file(url, dest_path):
    if not dest_path.exists():
        print(f"Downloading {dest_path.name}...")
        urllib.request.urlretrieve(url, dest_path)
        print(f"✓ Downloaded to {dest_path}")
    else:
        print(f"✓ Already exists: {dest_path.name}")

download_file(MODEL_BASE_URL + MODEL_PB_NAME, pb_path)
download_file(MODEL_BASE_URL + MODEL_JSON_NAME, json_path)

# Load class labels (the 40 instruments)
with open(json_path, "r") as f:
    model_meta = json.load(f)

INSTRUMENT_CLASSES = model_meta.get("classes", [])
print(f"Loaded {len(INSTRUMENT_CLASSES)} instrument classes:")
print(INSTRUMENT_CLASSES)


## 4. Initialize Essentia Inference Algorithms

We create two model instances using `TensorflowPredictMusiCNN`:
1. **`prob_model`**: Taps into `model/Sigmoid` to output the **40 instrument probabilities**.
2. **`embed_model`**: Taps into the penultimate layer to output **200-d dense embeddings**.

In [ ]:
# 1. Model for 40-class multi-label instrument probabilities (Sigmoid output)
prob_model = es.TensorflowPredictMusiCNN(
    graphFilename=str(pb_path),
    output="model/Sigmoid"
)

# 2. Model for 200-dimensional dense feature embeddings (penultimate layer)
# MusiCNN uses 'model/dense/BiasAdd' or 'model/dense_1/BiasAdd' as dense representation
try:
    embed_model = es.TensorflowPredictMusiCNN(
        graphFilename=str(pb_path),
        output="model/dense/BiasAdd"
    )
except Exception:
    # Fallback to default output layer if custom layer name differs
    embed_model = prob_model

print("✓ Essentia MusiCNN models initialized successfully.")


## 5. Single Track Feature Extraction Function

Essentia takes 16 kHz mono audio and automatically extracts 3-second mel-spectrogram patches. We aggregate patch-level predictions into a single song-level vector using mean pooling.

In [ ]:
def extract_track_features(audio_path, sample_rate=16000):
    """
    Loads an audio file and extracts:
    - patch_probs: (N_patches, 40) patch-level instrument probabilities
    - song_probs: (40,) mean-pooled song-level instrument probabilities
    - song_embed: (200,) mean-pooled dense embedding
    """
    try:
        # Load 16kHz mono audio via Essentia MonoLoader
        loader = es.MonoLoader(filename=str(audio_path), sampleRate=sample_rate)
        audio = loader()
        
        # Ensure audio has sufficient length (minimum 3 seconds / 48000 samples)
        if len(audio) < sample_rate * 3:
            audio = np.pad(audio, (0, sample_rate * 3 - len(audio)))
            
        # Run inference
        patch_probs = prob_model(audio)          # Shape: (N_patches, 40)
        patch_embeds = embed_model(audio)        # Shape: (N_patches, 200)
        
        # Pool across time patches to get song-level representations
        song_probs = np.mean(patch_probs, axis=0)      # Shape: (40,)
        song_embed = np.mean(patch_embeds, axis=0)    # Shape: (200,)
        
        return patch_probs, song_probs, song_embed
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None, None, None


## 6. Demonstration on a Sample Track & Temporal Analysis

Let's test the model on a sample audio file to visualize both the song-level instrument predictions and their temporal progression over 3-second windows.

In [ ]:
# Find any available audio file in AUDIO_DIR
sample_audio_files = list(AUDIO_DIR.rglob("*.mp3")) + list(AUDIO_DIR.rglob("*.wav"))

if sample_audio_files:
    sample_path = sample_audio_files[0]
    print(f"Analyzing sample: {sample_path.name}")
    
    patch_probs, song_probs, song_embed = extract_track_features(sample_path)
    
    if song_probs is not None:
        # Top 10 predicted instruments
        top_indices = np.argsort(song_probs)[::-1][:10]
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        
        # Bar chart: Song-level top predicted instruments
        sns.barplot(
            x=song_probs[top_indices],
            y=[INSTRUMENT_CLASSES[i] for i in top_indices],
            palette="mako",
            ax=axes[0]
        )
        axes[0].set_title(f"Top-10 Predicted Instruments: {sample_path.name}")
        axes[0].set_xlabel("Probability (Sigmoid)")
        axes[0].set_xlim(0, 1.0)
        
        # Heatmap: Patch-level temporal activation for top instruments
        top_patch_matrix = patch_probs[:, top_indices].T
        sns.heatmap(
            top_patch_matrix,
            yticklabels=[INSTRUMENT_CLASSES[i] for i in top_indices],
            cmap="inferno",
            cbar_kws={"label": "Probability"},
            ax=axes[1]
        )
        axes[1].set_title("Temporal Instrument Activations (3s Patches)")
        axes[1].set_xlabel("Patch Index (3s windows)")
        
        plt.tight_layout()
        plt.show()
else:
    print("No sample audio found in AUDIO_DIR yet. Skipping demo plot.")


## 7. Batch Extract Features for the Entire Dataset

We now iterate through all tracks in `song_manifest.csv` (or the audio directory) to extract both the **200-d embeddings** and **40-d probabilities**.

In [ ]:
embeddings_list = []
probabilities_list = []
processed_ids = []

# Check if manifest exists; otherwise scan AUDIO_DIR directly
if MANIFEST.exists():
    manifest_df = pd.read_csv(MANIFEST)
    track_list = []
    for _, row in manifest_df.iterrows():
        sid = str(row["song_id"])
        # Check possible audio file paths
        audio_candidates = list(AUDIO_DIR.glob(f"*{sid}*"))
        if audio_candidates:
            track_list.append((sid, audio_candidates[0]))
else:
    audio_files = sorted(list(AUDIO_DIR.rglob("*.mp3")) + list(AUDIO_DIR.rglob("*.wav")))
    track_list = [(f.stem, f) for f in audio_files]

print(f"Total tracks to process: {len(track_list)}")

for sid, audio_path in tqdm(track_list, desc="Extracting Essentia Instrument Features"):
    patch_p, song_p, song_e = extract_track_features(audio_path)
    if song_e is not None:
        embeddings_list.append(song_e)
        probabilities_list.append(song_p)
        processed_ids.append(sid)

if embeddings_list:
    E_matrix = np.array(embeddings_list, dtype=np.float32)
    P_matrix = np.array(probabilities_list, dtype=np.float32)
    
    # Save in standard format for Stage 2 Fusion (07_fusion_genre_classifier.ipynb)
    np.save(FEAT_DIR / "instrument_embeddings.npy", E_matrix)
    np.save(FEAT_DIR / "instrument_probabilities.npy", P_matrix)
    (FEAT_DIR / "song_ids.json").write_text(json.dumps(processed_ids, indent=2))
    (FEAT_DIR / "instrument_tags.json").write_text(json.dumps(INSTRUMENT_CLASSES, indent=2))
    
    print("\n" + "="*60)
    print(f"✓ SUCCESS: Extracted features for {len(processed_ids)} songs")
    print(f"  - Embeddings matrix:     {E_matrix.shape}  --> {FEAT_DIR / 'instrument_embeddings.npy'}")
    print(f"  - Probabilities matrix:  {P_matrix.shape}  --> {FEAT_DIR / 'instrument_probabilities.npy'}")
    print(f"  - Song IDs:              {len(processed_ids)} entries  --> {FEAT_DIR / 'song_ids.json'}")
    print("="*60)
else:
    print("No tracks processed. Please check your audio directory paths.")


## 8. Quantitative Evaluation against MTG-Jamendo Test Ground Truth

If test annotations are available in `annotations/splits/split-0/autotagging_instrument-test.tsv`, we evaluate the Essentia pre-trained predictions using **Macro ROC-AUC** and **Macro PR-AUC / mAP**.

In [ ]:
test_tsv = ANN_DIR / "splits" / "split-0" / "autotagging_instrument-test.tsv"

if test_tsv.exists() and probabilities_list:
    # Load ground truth binary matrix
    test_df = pd.read_csv(test_tsv, sep="\t")
    
    id_to_pred_idx = {sid: idx for idx, sid in enumerate(processed_ids)}
    
    y_true_list = []
    y_pred_list = []
    
    for _, row in test_df.iterrows():
        sid = f"{int(row['TRACK_ID']):07d}"
        if sid in id_to_pred_idx:
            # Parse ground truth tags
            tags = [t.replace("instrument---", "") for t in row["TAGS"].split(",")]
            target = np.zeros(len(INSTRUMENT_CLASSES), dtype=np.float32)
            for t in tags:
                if t in INSTRUMENT_CLASSES:
                    target[INSTRUMENT_CLASSES.index(t)] = 1.0
                    
            y_true_list.append(target)
            y_pred_list.append(P_matrix[id_to_pred_idx[sid]])
            
    if y_true_list:
        Y_true = np.array(y_true_list)
        Y_pred = np.array(y_pred_list)
        
        roc_scores = []
        pr_scores = []
        for c in range(Y_true.shape[1]):
            if Y_true[:, c].sum() > 0 and Y_true[:, c].sum() < len(Y_true):
                roc_scores.append(roc_auc_score(Y_true[:, c], Y_pred[:, c]))
                pr_scores.append(average_precision_score(Y_true[:, c], Y_pred[:, c]))
                
        macro_roc = np.mean(roc_scores)
        macro_pr = np.mean(pr_scores)
        
        print("\n" + "="*40)
        print("ESSENTIA TEST EVALUATION RESULTS")
        print("="*40)
        print(f"Evaluated on {len(Y_true)} test tracks")
        print(f"Macro ROC-AUC: {macro_roc:.4f}")
        print(f"Macro PR-AUC (mAP): {macro_pr:.4f}")
        print("="*40)
    else:
        print("No overlapping test tracks found.")
else:
    print("Test annotation file not found or no predictions generated.")


## 9. Next Steps

Your instrument features are now saved in `features/instrument/`.
- Proceed directly to **`04_rhythm_features.ipynb`**, **`05_timbre_features.ipynb`**, and **`06_harmony_features.ipynb`** to extract the other three musical concept spaces.
- Finally, run **`07_fusion_genre_classifier.ipynb`** to train the Cross-Concept Attention Fusion model and predict genre tags with built-in interpretability.